In [1]:
from app.model.model import TwoTowerModel, PlaceTower, UserTower
from app.model.contrastive_loss import ContrastiveLoss
from app.model.TwoTowerDataset import TwoTowerDataset, collate_fn, pad_cuisine, UserEmbeddingDataset, PlaceEmbeddingDataset, user_collate_fn, place_collate_fn
import torch
import torch.nn as nn

In [2]:
from torchview import draw_graph

In [3]:
# Helper to find max ID + 1 for Embedding layers
def get_vocab_size(dataset, tower_key, feature_key):
    max_id = 0
    for i in range(len(dataset)):
        val = dataset.data[i] # Access raw row
        # Note: adjust index based on SQL position if needed, 
        # but easier to use the dataset's __getitem__ keys:
        item = dataset[i][tower_key][feature_key]
        if isinstance(item, list):
            if item: max_id = max(max_id, max(item))
        else:
            max_id = max(max_id, item)
    return int(max_id + 1)

In [4]:
from torch.utils.data import DataLoader
import psycopg2
from torch.utils.data import random_split
conn = psycopg2.connect(database='reco_db', user='reco_user',password='reco_pass')
dataset = TwoTowerDataset(conn)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # reproducible
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)




OperationalError: connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?


In [ ]:
# loading num_cuisines from data
user_cuisines = set()
place_cuisines = set()

for i in range(len(dataset)):
    item = dataset[i]
    
    # Handle User Cuisines
    u_c = item['user']['cuisine_ids']
    if isinstance(u_c, list):
        user_cuisines.update(u_c)
    else:
        user_cuisines.add(u_c)
        
    # Handle Place Cuisines
    p_c = item['place']['cuisine_ids']
    if isinstance(p_c, list):
        place_cuisines.update(p_c)
    else:
        place_cuisines.add(p_c)

NUM_USERCUISINES = len(user_cuisines)
NUM_CUISINES = len(place_cuisines)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# load num_cuisines from data
BERT_DIM = 1536 
HIDDEN_DIM = 128

model_config = {
    "user_tower": {
        "numeric_dim": 3,
        "ordinal_dim": 3,
        "cuisine_vocab_size": NUM_USERCUISINES,
        "bert_dim": BERT_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dress_vocab": get_vocab_size(dataset, "user", "dress_preference_id"),
        "activity_vocab": get_vocab_size(dataset, "user", "activity_id"),
        "ambience_vocab": get_vocab_size(dataset, "user", "ambience_id"),
        "hijos_vocab": get_vocab_size(dataset, "user", "hijos_id"),
        "interest_vocab": get_vocab_size(dataset, "user", "interest_id"),
        "marital_vocab": get_vocab_size(dataset, "user", "marital_status_id"),
        "personality_vocab": get_vocab_size(dataset, "user", "personality_id"),
        "religion_vocab": get_vocab_size(dataset, "user", "religion_id"),
        "transport_vocab": get_vocab_size(dataset, "user", "transport_id"),
    },

    "place_tower": {
        "numeric_dim": 12,
        "ordinal_dim": 3,
        "cuisine_vocab_size": NUM_CUISINES,
        "bert_dim": BERT_DIM,
        "hidden_dim": HIDDEN_DIM,
        "smoking_vocab": get_vocab_size(dataset, "place", "smoking_area_id"),
        "rambience_vocab": get_vocab_size(dataset, "place", "rambience_id"),
        "parking_vocab": get_vocab_size(dataset, "place", "parking_lot_id"),
    }
}



user_tower =UserTower(**model_config["user_tower"]).to(device)

place_tower = PlaceTower(**model_config["place_tower"]).to(device)

torch.save({
    "user_tower_state": user_tower.state_dict(),
    "place_tower_state": place_tower.state_dict(),
    "config": model_config
}, "models/twotower/twotower_model.pt")

model = TwoTowerModel(user_tower, place_tower)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



In [ ]:
dummy_input = torch.randn(1, 3, 224, 224) 

model_graph = draw_graph(model, input_size=dummy_input.shape, expand_nested=True)
# To display the graph (e.g., in a Jupyter notebook)
model_graph.visual_graph 
# To save the visualization as a file (e.g., PNG or SVG)
model_graph.visual_graph.render("resnet18_graph", format="png") 